# Hướng dẫn sử dụng thư viện gex-msgraph

Notebook này hướng dẫn bạn sử dụng thư viện `gex-msgraph` từng bước một — từ kết nối SharePoint, đọc file, quản lý thư mục, đến gửi tin nhắn. Hãy đảm bảo file `.env` của bạn đã điền đầy đủ thông tin `MS_CUSTOM_*` trước khi chạy bất kỳ cell nào.

## 0. Cài đặt & Import

`%pip install` thêm thư viện vào kernel hiện tại. `load_dotenv()` đọc file `.env` để thông tin đăng nhập (credentials) được nạp vào biến môi trường trước khi bạn tạo client.

In [1]:
%pip install --no-cache-dir git+https://github.com/khanh-at-gex/gex-msgraph.git python-dotenv

  Cloning https://github.com/khanh-at-gex/gex-msgraph.git to C:\Users\khanh.nguyen-quoc\AppData\Local\Temp\pip-req-build-feoabm8d
  Resolved https://github.com/khanh-at-gex/gex-msgraph.git to commit f5b20c0c5f6377c1c7d7d6962f302f432f7bd141
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
Note: you may need to restart the kernel to use updated packages.


  Running command git clone --filter=blob:none --quiet https://github.com/khanh-at-gex/gex-msgraph.git 'C:\Users\khanh.nguyen-quoc\AppData\Local\Temp\pip-req-build-feoabm8d'

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Import necessary libraries
from dotenv import load_dotenv; load_dotenv()
from gex_msgraph import GraphClient, FileItem

## 1. Kết nối

`GraphClient("custom")` tự động lấy thông tin đăng nhập từ các biến môi trường `MS_CUSTOM_*` trong file `.env`. Chúng ta khởi tạo client một lần ở đây và dùng lại cho tất cả các cell tiếp theo — hãy chạy các cell theo thứ tự và đóng client khi hoàn tất.

In [3]:
client = GraphClient("custom")
await client.__aenter__()

items = await client.list_files("")
print(f"Connected. Found {len(items)} items at the drive root.")

Connected. Found 3 items at the drive root.


In [4]:
for item in items:
    print(item.name)

2026
Attachments
gex_test_folder


In [5]:
items2 = await client.list_files("gex_test_folder")
for item in items2:
    print(item.name)

test_01.csv
test_01.xlsx
test_02.csv
test_02.xlsx
test_remote.txt


In [6]:
items3 = await client.list_files("gex_test_folder")
for item in items3:
    label = "(folder)" if item.is_folder else f"{item.size:,} bytes"
    print(item.name, label)

test_01.csv 24 bytes
test_01.xlsx 8,393 bytes
test_02.csv 24 bytes
test_02.xlsx 8,393 bytes
test_remote.txt 15 bytes


## 2. Đọc File

### 2a. Excel theo đường dẫn (path)

`read_excel` tải file xuống và phân tích trực tiếp thành pandas DataFrame. `item_path` là đường dẫn đến file tính từ gốc SharePoint drive (không có dấu `/` ở đầu).

In [7]:
df = await client.read_excel(item_path="gex_test_folder/test_01.xlsx")
display(df.head())

,Col1,col2
0,A,1
1,B,2


### 2b. Excel theo share URL

Bạn có thể dùng link chia sẻ thay vì đường dẫn. Trong SharePoint, mở file → **Share** → **Copy link** để lấy URL. Dán trực tiếp vào tham số `share_url`.

In [8]:
df = await client.read_excel(share_url="https://gelexvn-my.sharepoint.com/:x:/g/personal/das-noreply_gelex_vn/IQBKtOdZeBotS7XJAfavXd0wAUkqK3K0PvXVSE9627CaMGU?e=fTQhM0")  # NOTE: replace with your actual value

display(df.head())


,Col1,col2
0,A,1
1,B,2


### 2c. CSV theo đường dẫn (path)

`read_csv` hoạt động giống `read_excel` — truyền `item_path` và nhận về một DataFrame. Các tham số bổ sung (ví dụ `sep=";"`) sẽ được chuyển tiếp thẳng vào `pandas.read_csv`.

In [9]:
df = await client.read_csv(item_path="gex_test_folder/test_01.csv")
display(df.head())


,Col1,col2
0,A,1
1,B,2


### 2d. Tải file dạng bytes thô

`download` trả về dữ liệu thô của file dưới dạng bytes — hữu ích cho ảnh, PDF, hoặc bất kỳ file nào không phải dạng bảng. Dùng lệnh `open` thông thường để ghi dữ liệu ra đĩa.

In [10]:
data = await client.download(item_path="gex_test_folder/test_01.csv")
print(f"Downloaded {len(data)} bytes")

with open("test_01.csv", "wb") as f:
    f.write(data)
print("Saved to test_01.csv")

Downloaded 24 bytes
Saved to test_01.csv


## 3. Sheet trong Excel

`list_excel_sheets` lấy danh sách tên sheet mà không cần tải toàn bộ file. Truyền tên sheet (hoặc chỉ số) vào `read_excel` qua tham số `sheet` để đọc đúng tab cần thiết.

In [11]:
sheets = await client.list_excel_sheets(item_path="gex_test_folder/test_01.xlsx")
print("Available sheets:", sheets)

df = await client.read_excel(item_path="gex_test_folder/test_01.xlsx", sheet=sheets[0])
df

Available sheets: ['Sheet1']


,Col1,col2
0,A,1
1,B,2


## 4. Đọc nhiều file cùng lúc

`read_excel_many` đọc nhiều file đồng thời và nối chúng thành một DataFrame duy nhất, tự động thêm cột `_source` để biết mỗi dòng dữ liệu đến từ file nào. Truyền `return_status=True` để nhận thêm bảng trạng thái cho biết file nào thành công, file nào bị bỏ qua.

In [12]:
df, status_df = await client.read_excel_many(
    ["gex_test_folder/test_01.xlsx", "gex_test_folder/test_02.xlsx"],
    on_error="skip",
    return_status=True,
)
display(df.head())
display(status_df)

,Col1,col2,_source
0,A,1,gex_test_folder/test_01.xlsx
1,B,2,gex_test_folder/test_01.xlsx
2,A,1,gex_test_folder/test_02.xlsx
3,B,2,gex_test_folder/test_02.xlsx


,path,status,error
0,gex_test_folder/test_01.xlsx,success,
1,gex_test_folder/test_02.xlsx,success,


## 5. Khám phá cấu trúc thư mục

Ba phương thức giúp bạn duyệt drive: `list_files` trả về các item con trực tiếp trong thư mục (không đệ quy), `walk` tìm kiếm đệ quy tất cả file khớp với glob pattern, và `get_folder_tree` tạo cây thư mục có thể in được cho toàn bộ cấu trúc.

In [13]:
items = await client.list_files("gex_test_folder")
for item in items:
    label = "(folder)" if item.is_folder else f"{item.size:,} bytes"
    print(item.name, label)

test_01.csv 24 bytes
test_01.xlsx 8,393 bytes
test_02.csv 24 bytes
test_02.xlsx 8,393 bytes
test_remote.txt 15 bytes


In [14]:
xlsx_files = await client.walk("gex_test_folder", pattern="*.xlsx")
print(f"Found {len(xlsx_files)} .xlsx files recursively")
for f in xlsx_files:
    print(" ", f.path)

Found 2 .xlsx files recursively
  gex_test_folder/test_01.xlsx
  gex_test_folder/test_02.xlsx


In [15]:
tree = await client.get_folder_tree("gex_test_folder")
tree.print()

📁 gex_test_folder (16849 bytes)
  📄 test_01.csv (24 bytes)
  📄 test_01.xlsx (8393 bytes)
  📄 test_02.csv (24 bytes)
  📄 test_02.xlsx (8393 bytes)
  📄 test_remote.txt (15 bytes)


## 6. Quản lý File

`DRY_RUN = True` (đặt ở cell tiếp theo) ngăn mọi thao tác ghi được thực thi — chỉ chuyển thành `False` khi bạn thực sự muốn thay đổi dữ liệu. `get_metadata` chỉ đọc và luôn chạy bất kể giá trị của DRY_RUN.

In [16]:
DRY_RUN = True  # Đổi thành False khi muốn thực sự ghi/gửi

meta = await client.get_metadata(item_path="gex_test_folder/test_01.xlsx")
print(f"Name:     {meta.name}")
print(f"Size:     {meta.size:,} bytes")
print(f"Modified: {meta.modified}")

Name:     test_01.xlsx
Size:     8,393 bytes
Modified: 2026-05-04 07:17:39+00:00


### Tạo thư mục & Upload file

`create_folder` tạo đường dẫn thư mục lồng nhau chỉ trong một lần gọi. `upload` truyền file local lên đường dẫn remote chỉ định — cả hai đường dẫn đều tính từ gốc drive.

In [17]:
if not DRY_RUN:
    folder = await client.create_folder("gex_test_folder/test_subfolder")  # NOTE: replace with your actual value
    print("Created folder:", folder.path)

    result = await client.upload("local_report.xlsx", "gex_test_folder/test_subfolder/report.xlsx")  # NOTE: replace with your actual value
    print("Uploaded:", result)
else:
    print("DRY_RUN: skipped create_folder and upload")

DRY_RUN: skipped create_folder and upload


### Di chuyển & Xóa file

`move_file` di chuyển file và có thể đổi tên cùng lúc. `delete_file` xóa file vĩnh viễn — không có thùng rác, vì vậy hãy kiểm tra kỹ đường dẫn trước khi đặt `DRY_RUN = False`.

In [18]:
if not DRY_RUN:
    moved = await client.move_file(
        "gex_test_folder/test_subfolder/report.xlsx",  # NOTE: replace with your actual value
        dest_folder_path="Archive",  # NOTE: replace with your actual value
        new_name="report_archived.xlsx",
    )
    print("Moved to:", moved.path)

    await client.delete_file(item_path="Archive/report_archived.xlsx")  # NOTE: replace with your actual value
    print("File deleted.")
else:
    print("DRY_RUN: skipped move_file and delete_file")

DRY_RUN: skipped move_file and delete_file


## 7. Giao tiếp

### 7a. Gửi email

`send_mail` gửi email dạng văn bản thuần từ hộp thư của service account. Đảm bảo `DRY_RUN = True` nếu chưa muốn gửi thật.

In [19]:
DRY_RUN = False  # Đổi thành True để chỉ in ra mà không thực hiện hành động nào

In [20]:
if not DRY_RUN:
    await client.send_mail(
        "lan.nguyen-ho@gelex.vn",  # NOTE: replace with your actual value
        "Subject: Test complete",
        "The daily pipeline finished successfully.",
    )
    print("Email sent.")
else:
    print("DRY_RUN: skipped send_mail")

Email sent.


### 7b. Teams chat — Liệt kê & Đọc tin nhắn

`list_chats()` trả về toàn bộ cuộc chat mà tài khoản service tham gia. `get_chat_messages(chat_id, limit)` lấy tin nhắn gần nhất từ một chat cụ thể.

In [21]:
chats = await client.list_chats()
print(f"Tìm thấy {len(chats)} cuộc chat\n")

for chat in chats:
    topic = chat.get("topic") or "(không có chủ đề)"
    chat_type = chat.get("chatType", "")
    members = [m.get("displayName", "?") for m in chat.get("members", [])]
    print(f"ID:         {chat['id']}")
    print(f"Loại:       {chat_type}")
    print(f"Chủ đề:     {topic}")
    print(f"Thành viên: {', '.join(members)}")
    print()

Tìm thấy 2 cuộc chat

ID:         19:10b8a68b5299413793129a7168b1e8b2@thread.v2
Loại:       group
Chủ đề:     DAS - Group BÁO động
Thành viên: Phòng Dữ liệu & Công nghệ QTRR - GELEX QTRR: DAS system, Nguyen Ho Lan, Nguyen Quoc Khanh

ID:         19:c5b31289-79e3-465b-a5b4-2830a974d949_c9989ac0-1331-4553-8a46-a2ef0bb7be42@unq.gbl.spaces
Loại:       oneOnOne
Chủ đề:     (không có chủ đề)
Thành viên: Phòng Dữ liệu & Công nghệ QTRR - GELEX QTRR: DAS system, Nguyen Quoc Khanh



In [22]:
chat_id = chats[0]["id"]  # NOTE: thay bằng ID chat cụ thể nếu cần

messages = await client.get_chat_messages(chat_id, limit=5)
for msg in messages:
    sender = (msg.get("from") or {}).get("user", {}).get("displayName", "?")
    body = msg.get("body", {}).get("content", "")
    created = msg.get("createdDateTime", "")
    print(f"[{created}] {sender}: {body}")

[2026-05-04T09:12:06.672Z] Phòng Dữ liệu & Công nghệ QTRR - GELEX QTRR: DAS system: Xin chào từ gex-msgraph!
[2026-05-04T09:02:07.151Z] Phòng Dữ liệu & Công nghệ QTRR - GELEX QTRR: DAS system: Xin chào từ gex-msgraph!
[2026-05-04T09:00:06.147Z] ?: <systemEventMessage/>
[2026-05-04T08:59:52.349Z] Nguyen Quoc Khanh: <p>thui để e kick a Thọ tạm để test các thứ anh ui</p>
[2026-05-04T08:59:11.071Z] Nguyen Xuan Tho: <p>wtf hello</p>


### 7c. Gửi tin nhắn vào chat

`send_chat_message(chat_id, text)` gửi tin nhắn trực tiếp vào một cuộc chat (1-1 hoặc nhóm). Dùng `chat_id` lấy từ bước 7b ở trên.

In [23]:
if not DRY_RUN:
    await client.send_chat_message(chat_id, "Xin chào từ gex-msgraph!")
    print("Đã gửi tin nhắn vào chat.")
else:
    print("DRY_RUN: bỏ qua send_chat_message")

Đã gửi tin nhắn vào chat.


## 8. Xử lý lỗi

Tất cả các lệnh gọi API đều đi qua `httpx`, nên mọi lỗi HTTP — 404 không tìm thấy, 403 bị từ chối, v.v. — sẽ ném ra ngoại lệ `httpx.HTTPStatusError`. Bắt ngoại lệ này để xem mã trạng thái và nội dung phản hồi.

In [24]:
import httpx

try:
    df = await client.read_excel(item_path="does/not/exist.xlsx")
except httpx.HTTPStatusError as e:
    print(e.response.status_code, e.response.text)

Graph request failed 404: {"error":{"code":"itemNotFound","message":"The resource could not be found."}}


404 {"error":{"code":"itemNotFound","message":"The resource could not be found."}}


## Dọn dẹp

Đóng client khi hoàn tất để giải phóng connection pool HTTP bên dưới.

In [25]:
await client.__aexit__(None, None, None)
print("Client closed.")

Client closed.
